In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.veiculos_eletricos_raw;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.veiculos_eletricos_raw.arquivos_senatran
COMMENT 'Arquivos brutos originais da SENATRAN utilizados no projeto de veículos elétricos';

In [0]:
import os

caminho_raw = "/Volumes/workspace/veiculos_eletricos_raw/arquivos_senatran/frota_combustivel/2026/07"

os.makedirs(caminho_raw, exist_ok=True)

print(caminho_raw)

In [0]:
arquivos = os.listdir(caminho_raw)

for arquivo in arquivos:
    print(arquivo)

In [0]:
%pip install openpyxl


In [0]:

import pandas as pd

caminho_arquivo = (
    f"{caminho_raw}/"
    "copy_of_D_Frota_por_UF_Municipio_COMBUSTIVEL_Julho_2026.xlsx"
)

df_pandas = pd.read_excel(
    caminho_arquivo,
    sheet_name="Layout D "
)

print(df_pandas.shape)

display(df_pandas.head(10))

In [0]:
arquivo_excel = pd.ExcelFile(caminho_arquivo)
print(arquivo_excel.sheet_names)

In [0]:
df_spark = spark.createDataFrame(df_pandas)

df_spark.printSchema()

display(df_spark.limit(10))

In [0]:
from pyspark.sql import functions as F
import os

nome_arquivo = os.path.basename(caminho_arquivo)

df_com_metadados = (
    df_spark
    .withColumn("source_file", F.lit(nome_arquivo))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("reference_year", F.lit(2026).cast("int"))
    .withColumn("reference_month", F.lit(7).cast("int"))
)

df_com_metadados.printSchema()

display(df_com_metadados.limit(10))

In [0]:
print("Quantidade de linhas:")
print(df_com_metadados.count())

print("\nCombustíveis distintos:")
print(
    df_com_metadados
    .select("Combustível Veículo")
    .distinct()
    .count()
)

print("\nPeríodo de referência:")
df_com_metadados \
    .select("reference_year", "reference_month") \
    .distinct() \
    .show()